# Pipeline de Transformação: vendas

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from pyspark.sql import functions as F
from src.modules.spark_session import get_spark_session, close_spark_session
import src.modules.transform as transform
import src.modules.transform_vendas as transform_vendas

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("TransformVendas")

# Leitura dos dados da camada Bronze

In [ ]:
# Caminho da tabela Bronze no MinIO
bronze_path = "s3a://bronze/vendas"

# Lê os dados originais da Bronze
df_vendas = spark.read.parquet(bronze_path)

print(f"Total de registros carregados: {df_vendas.count()}")
df_vendas.printSchema()
df_vendas.limit(5).toPandas()

# Aplica TRIM nas colunas de texto

In [ ]:
text_cols = [
    "produto", "categoria", "marca", "canal_venda", 
    "forma_pagamento", "cidade", "estado", "status_pedido"
]
df_vendas = transform.trim_columns(df_vendas, text_cols)

df_vendas.select("pedido_id", "produto", "cidade").limit(5).toPandas()

# Preenche valores nulos por 0 nas colunas de valor

In [ ]:
value_cols = [
    "preco_unitario", "desconto", "frete", "valor_total"
]

df_vendas = transform.fill_null_values(df_vendas, value_cols, fill_value=0.0)

df_vendas.select("pedido_id", "preco_unitario", "desconto", "frete", "valor_total").limit(5).toPandas()

# Cria a coluna 'valor_total_sem_desconto'

In [ ]:
df_vendas = transform_vendas.create_total_without_discount(df_vendas)

df_vendas.select("pedido_id", "preco_unitario", "quantidade", "frete", "desconto", "valor_total", "valor_total_sem_desconto").limit(5).toPandas()

# Capitaliza as colunas de texto

In [ ]:
df_vendas = transform.capitalize_columns(df_vendas, ["cidade"])

df_vendas.select("pedido_id", "cidade").limit(5).toPandas()

# Criar colunas de ano, mês, dia

In [ ]:
df_vendas = transform.extract_date_parts(df_vendas, "data_pedido")

df_vendas.select("pedido_id", "data_pedido", "ano", "mes", "dia").limit(5).toPandas()

# Arredondar as colunas de valor para 2 casas decimais

In [ ]:
all_value_cols = value_cols + ["valor_total_sem_desconto"]
df_vendas = transform.round_values(df_vendas, all_value_cols, decimals=2)

df_vendas.select("pedido_id", "preco_unitario", "desconto", "frete", "valor_total", "valor_total_sem_desconto").limit(5).toPandas()

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)